In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#logistic regresssion from scratch using pandas matplot and numpy

In [52]:
#loading dataset and EDA
train=pd.read_csv(r"C:\Users\Samaira Singh\Downloads\bootcamp\ml\train.csv")
test=pd.read_csv(r"C:\Users\Samaira Singh\Downloads\bootcamp\ml\test.csv")
res=pd.read_csv(r"C:\Users\Samaira Singh\Downloads\bootcamp\ml\gender_submission.csv")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:\n", train_df.isnull().sum()[train_df.isnull().sum() > 0])

train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train["IsAlone"]    = (train["FamilySize"] == 1).astype(int)
test["FamilySize"] = test["SibSp"] + test_df["Parch"] + 1
test["IsAlone"]    = (test["FamilySize"] == 1).astype(int)
 
 

Train shape: (891, 14)
Test shape : (418, 13)

Missing values in train:
 Age         177
Cabin       687
Embarked      2
dtype: int64


In [53]:
#feauture selection to prevent overfit
#dropping the survived column helps in preventing data leakage
features = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'SibSp',
    'Parch'
]

#data cleaning 
#missing age replaced with median of all the ages in the column and same goes for missing fare and missing  fare
train.fillna({"Age":train["Age"].median()},inplace =True)
test.fillna({"Age":test["Age"].median()},inplace =True)
test.fillna({"Fare":test["Fare"].median()},inplace =True)
#encoding
#categorical data converted to number so that it can be multiplied with weights 

train["Sex"] = train["Sex"].str.lower()

train["Sex"] = train["Sex"].map({
    "male":0,
    "female":1
})
test["Sex"] = test["Sex"].str.lower()

test["Sex"] = test["Sex"].map({
    "male":0,
    "female":1
})


X_train = train[features].values
y_train = train['Survived'].values

X_test = test[features].values

In [54]:
# feature scaling 
#prevents overfit and disproportional influence of larger features that can reduce accuracy of model
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


In [55]:
#adding bias column 
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]
#sigmoid function 
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1/(1 + np.exp(-z))
#initialising parameters
m, n = X_train.shape

weights = np.zeros(n)

In [89]:
#loss
def compute_loss(y, y_pred):

    epsilon = 1e-15

    y_pred = np.clip(
        y_pred,
        epsilon,
        1-epsilon
    )

    return -np.mean(
        y*np.log(y_pred)
        +
        (1-y)*np.log(1-y_pred)
    )
#gradient descent
#similar to linear regression 
learning_rate = 0.05
epochs = 5000

for epoch in range(epochs):

    z = np.dot(X_train, weights)

    y_pred = sigmoid(z)

    gradient = (
        np.dot(
            X_train.T,
            (y_pred - y_train)
        )
        / m
    )

    weights -= learning_rate * gradient

    if epoch % 500 == 0:

        loss = compute_loss(
            y_train,
            y_pred
        )

        print(
            f"Epoch {epoch}, Loss={loss:.4f}"
        )

Epoch 0, Loss=0.4429
Epoch 500, Loss=0.4429
Epoch 1000, Loss=0.4429
Epoch 1500, Loss=0.4429
Epoch 2000, Loss=0.4429
Epoch 2500, Loss=0.4429
Epoch 3000, Loss=0.4429
Epoch 3500, Loss=0.4429
Epoch 4000, Loss=0.4429
Epoch 4500, Loss=0.4429


In [90]:
#prediction
#threshold set to 0.55
#raining threshold eliminates false positives
probabilities = sigmoid(
    np.dot(X_test, weights)
)

predictions = (
    probabilities >= 0.55
).astype(int)

submission = pd.DataFrame({

    'PassengerId':
        test['PassengerId'],

    'Survived':
        predictions
})

submission.to_csv(
    'submission.csv',
    index=False
)

In [91]:
#accuracy
y_test = res["Survived"].values
accuracy = np.mean(predictions == y_test)
print(f"Accuracy = {accuracy*100:.2f}%")

Accuracy = 94.98%


In [92]:
#confusion matrix 
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

[[260   6]
 [ 15 137]]


In [93]:
'''Only 21 wrong predictions out of 418.The model is slightly better at identifying who died (260 correct, only 6 missed) than who survived
(137 correct, 15 missed) — which makes sense since the training data has more deaths than survivors.'''

'Only 21 wrong predictions out of 418.The model is slightly better at identifying who died (260 correct, only 6 missed) than who survived\n(137 correct, 15 missed) — which makes sense since the training data has more deaths than survivors.'

In [94]:
print(probabilities[:20])

[0.08172875 0.37711034 0.08242786 0.10705038 0.5966333  0.16694215
 0.6267993  0.19908659 0.7291171  0.07321394 0.10684199 0.3439862
 0.94442236 0.0599982  0.86142974 0.82501875 0.20775939 0.13136612
 0.57169275 0.48160574]


In [95]:
print(weights)

[-0.64563624 -0.90866352  1.31893021 -0.51266208  0.14134242 -0.38440449
 -0.08596564]
